# 3. Data Preparation

In [1]:
# importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet
from scipy.spatial.distance import pdist
import os
import joblib

In [2]:
df = pd.read_csv('../data/HotelCustomersDataset.tsv', sep='\t')
print(f'Starting shape: {df.shape}')

Starting shape: (83590, 31)


## 3.1 Select Data

This section defines which rows and columns enter the clustering dataset and explains the reasoning for each decision.

### Row selection

**Ghost customers (19,920 rows): excluded.** These customers have no stay activity whatsoever: zero lodging revenue, zero bookings, zero room nights, zero person nights. They have never interacted with the hotel in a measurable way. Keeping them would produce a cluster with no marketing interpretation.

**Customers under 18: excluded.** These are most likely child companions alongside an adult booking holder. The booking and spending decision is made by adults, so including under-18 observations would introduce records that are not the target of any marketing action.

**All other customers: included**, including complementary stays (488 rows). Zero lodging revenue in that group is structural: they stayed, and their behavioural fields carry valid information.

### Column selection

**`ID`, `NameHash`, `DocIDHash`: dropped after duplicate handling and merging.** These are needed to identify exact duplicates and to merge split profiles. They are dropped only once they are no longer needed.

**Low-variance SR flags: excluded.** 9 of 13 special request variables have a presence below 1%. They add noise without contributing to the segmentation.

**Retained SR flags**: `SRHighFloor` (4.5%), `SRKingSizeBed` (35%), `SRTwinBed` (14%), `SRQuietRoom` (9%).

**All behavioural variables retained**: revenue, bookings, nights, recency, lead time, tenure, channel, segment, nationality. These are transformed in section 3.3.

In [3]:
# removing "ghost" customers with no stay activity
ghost_mask = (
    (df['LodgingRevenue'] == 0) &
    (df['BookingsCheckedIn'] == 0) &
    (df['RoomNights'] == 0) &
    (df['PersonsNights'] == 0)
)
df = df[~ghost_mask].copy()
print(f'Ghost customers removed: {ghost_mask.sum():,}')
print(f'Remaining: {len(df):,}')

Ghost customers removed: 19,920
Remaining: 63,670


## 3.2 Clean Data

Cleaning follows the order below because each step affects the data used by the next.

The identifier columns `ID`, `NameHash`, and `DocIDHash` are retained through steps A and B because they are needed for duplicate detection and profile merging. They are dropped in step C once they are no longer needed.

### A. Remove Exact Duplicate Profiles

Exact duplicates are checked across all columns except `ID`, which is unique across the data. Records identical on every other field are assumed to be the same customer profile entered more than once. The first occurrence is kept.

Exact duplicate removal is performed after ghost customer removal, so the number of duplicates removed here is lower than the total count identified in Data Understanding (80). Many of those duplicates were ghost customers and have already been removed in the previous step.

In [4]:
cols_without_id = [col for col in df.columns if col != 'ID']

n_before = len(df)
df = df.drop_duplicates(subset=cols_without_id, keep='first')

print(f'Rows removed: {n_before - len(df)}')
print(f'Remaining: {len(df):,}')

Rows removed: 16
Remaining: 63,654


### B. Merge Split Profiles

Rows sharing both `NameHash` and `DocIDHash` but with different behavioural fields represent the same real customer whose data has been split across multiple PMS profiles and are merged.

After merging, the unit of analysis changes: each row represents **one real customer**.

The `-1` sentinel in `DaysSinceLastStay` and `DaysSinceFirstStay` is already absent after ghost removal (verified: 0 rows with -1), so `min` and `max` aggregation is safe.

Aggregation rules:
- **Sum**: `LodgingRevenue`, `OtherRevenue`, `BookingsCanceled`, `BookingsNoShowed`, `BookingsCheckedIn`, `PersonsNights`, `RoomNights`
- **Min**: `DaysSinceLastStay`: lower value means more recent stay
- **Max**: `DaysSinceFirstStay`, `DaysSinceCreation`: higher value captures the earliest record
- **Mean**: `Age`, `AverageLeadTime`
- **Max**: all SR flags. If either profile has the preference, the customer has it
- **Mode**: `Nationality`, `MarketSegment`, `DistributionChannel`
- **First**: `ID`. Retained temporarily, but dropped in step C

In [5]:
sr_cols = [col for col in df.columns if col.startswith('SR')]

# confirm -1 sentinels are absent before aggregating recency fields
print(f'DaysSinceLastStay == -1: {(df["DaysSinceLastStay"] == -1).sum()}')
print(f'DaysSinceFirstStay == -1: {(df["DaysSinceFirstStay"] == -1).sum()}')

DaysSinceLastStay == -1: 0
DaysSinceFirstStay == -1: 0


In [6]:
agg_dict = {
    'ID': 'first',
    'Age': 'mean',
    'DaysSinceCreation': 'max',
    'AverageLeadTime': 'mean',
    'LodgingRevenue': 'sum',
    'OtherRevenue': 'sum',
    'BookingsCanceled': 'sum',
    'BookingsNoShowed': 'sum',
    'BookingsCheckedIn': 'sum',
    'PersonsNights': 'sum',
    'RoomNights': 'sum',
    'DaysSinceLastStay': 'min',
    'DaysSinceFirstStay': 'max',
    'Nationality': lambda x: x.mode()[0],
    'MarketSegment': lambda x: x.mode()[0],
    'DistributionChannel': lambda x: x.mode()[0],
}
for col in sr_cols:
    agg_dict[col] = 'max'

n_before = len(df)
df = df.groupby(['NameHash', 'DocIDHash'], as_index=False).agg(agg_dict)

print(f'Rows consolidated: {n_before - len(df):,}')
print(f'Remaining: {len(df):,}')

Rows consolidated: 1,801
Remaining: 61,853


### C. Drop Identifiers and Low-Variance Columns

Now that exact duplicates have been removed and split profiles merged, `ID`, `NameHash`, and `DocIDHash` are no longer needed and are dropped. Low-variance SR columns are also dropped at this point.

In [7]:
# drop identifiers
df = df.drop(columns=['ID', 'NameHash', 'DocIDHash'])

# drop near-zero variance SR columns
low_var_sr = ['SRLowFloor', 'SRAccessibleRoom', 'SRMediumFloor', 'SRBathtub','SRShower', 'SRCrib', 'SRNearElevator', 'SRAwayFromElevator', 'SRNoAlcoholInMiniBar']
df = df.drop(columns=low_var_sr)

print(f'Columns after dropping identifiers and low-variance SR: {df.shape[1]}')

Columns after dropping identifiers and low-variance SR: 19


### D. Treat Age

Three age issues are addressed, each with a different rationale:

- **Age < 0 and Age > 100**: treated as implausible values and removed.
- **Age 0–17** (931 rows): while these may be valid records, customers under 18 are unlikely to be the booking or spending decision-maker. Segmenting them alongside adult customers would produce segments that cannot be meaningfully targeted by the marketing department. Removed as a business decision.
- **Missing Age**: imputed with the median computed after the above removals.

In [8]:
n_before = len(df)
df = df[df['Age'].isna() | ((df['Age'] >= 18) & (df['Age'] <= 100))]
print(f'Rows removed (implausible or under-18): {n_before - len(df)}')

age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

print(f'Missing age imputed with median: {age_median}')
print(f'Missing values remaining: {df["Age"].isnull().sum()}')
print(f'Remaining: {len(df):,}')

Rows removed (implausible or under-18): 950
Missing age imputed with median: 48.0
Missing values remaining: 0
Remaining: 60,903


### E. Treat AverageLeadTime -1 Value

`AverageLeadTime = -1` is not a real lead time. Recoded to `NaN` and imputed with the median.

In [9]:
df['AverageLeadTime'] = df['AverageLeadTime'].replace(-1, np.nan)

leadtime_median = df['AverageLeadTime'].median()
df['AverageLeadTime'] = df['AverageLeadTime'].fillna(leadtime_median)

print(f'Imputed with median: {leadtime_median}')
print(f'Missing values remaining: {df["AverageLeadTime"].isnull().sum()}')

Imputed with median: 59.0
Missing values remaining: 0


### F. Cap Outliers

No rows are removed, capping is used. Thresholds are computed on the fully cleaned population at this point.

- **p99**: `LodgingRevenue`, `OtherRevenue`, `AverageLeadTime`
- **p99.9**: `BookingsCheckedIn`, `RoomNights`, `PersonsNights`, since p99 is too low for these variables (2, 8, 20) because the distribution is compressed by one-time visitors

In [10]:
cap_p99  = ['LodgingRevenue', 'OtherRevenue', 'AverageLeadTime']
cap_p999 = ['BookingsCheckedIn', 'RoomNights', 'PersonsNights']

cap_summary = []
for col in cap_p99:
    cap = df[col].quantile(0.99)
    n_capped = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    cap_summary.append({'Variable': col, 'Threshold': 'p99', 'Cap Value': round(cap, 2), 'Values Capped': n_capped})

for col in cap_p999:
    cap = df[col].quantile(0.999)
    n_capped = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    cap_summary.append({'Variable': col, 'Threshold': 'p99.9', 'Cap Value': round(cap, 2), 'Values Capped': n_capped})

display(pd.DataFrame(cap_summary))

,Variable,Threshold,Cap Value,Values Capped
0,LodgingRevenue,p99,1959.87,610
1,OtherRevenue,p99,530.49,610
2,AverageLeadTime,p99,423.00,558
3,BookingsCheckedIn,p99.9,8.00,59
4,RoomNights,p99.9,22.00,59
5,PersonsNights,p99.9,36.00,50


### G. Cleaning Verification

Confirming all issues identified in section 2.4 have been resolved.

In [11]:
cleaning_check = pd.DataFrame({
    'Check': [
        'Age missing', 'Age < 0', 'Age > 100', 'Age < 18',
        'AverageLeadTime missing', 'AverageLeadTime < 0',
        'DaysSinceLastStay = -1', 'DaysSinceFirstStay = -1'
    ],
    'Count': [
        df['Age'].isna().sum(),
        (df['Age'] < 0).sum(),
        (df['Age'] > 100).sum(),
        (df['Age'] < 18).sum(),
        df['AverageLeadTime'].isna().sum(),
        (df['AverageLeadTime'] < 0).sum(),
        (df['DaysSinceLastStay'] == -1).sum(),
        (df['DaysSinceFirstStay'] == -1).sum()
    ]
})

display(cleaning_check)

,Check,Count
0,Age missing,0
1,Age < 0,0
2,Age > 100,0
3,Age < 18,0
4,AverageLeadTime missing,0
5,AverageLeadTime < 0,0
6,DaysSinceLastStay = -1,0
7,DaysSinceFirstStay = -1,0


## 3.3 Construct Data

Three new variables are derived to represent customer value, loyalty, and demographic profile. All other engineered features tested during modelling exploration (revenue per night, persons per room, other revenue ratio, log transforms) were found to weaken cluster separation and are not included. The reasoning for each decision is documented below.

### A. Total Revenue

`TotalRevenue` is computed as `LodgingRevenue + OtherRevenue`. It replaces both raw revenue columns as the single monetary signal for the clustering model. The composition (`OtherRevenue / TotalRevenue`) is retained in the profile dataset as a descriptor.

`LodgingRevenue` and `OtherRevenue` are dropped after this step.

In [12]:
df['TotalRevenue'] = df['LodgingRevenue'] + df['OtherRevenue']

df = df.drop(columns=['LodgingRevenue', 'OtherRevenue'])

print(f'TotalRevenue: median={df["TotalRevenue"].median():.1f}, max={df["TotalRevenue"].max():.1f}')
print(f'Columns remaining: {df.shape[1]}')

TotalRevenue: median=384.0, max=2490.4
Columns remaining: 18


### B. Customer Tenure

`Tenure` is computed as `DaysSinceFirstStay − DaysSinceLastStay`. It represents the spread between a customer's first and most recent stay, a measure of loyalty and repeat behaviour that neither raw variable captures alone.

A customer with `DaysSinceFirstStay = 500` and `DaysSinceLastStay = 490` has Tenure = 10: effectively a one-time visitor whose first and only stay was 490 days ago. A customer with `DaysSinceFirstStay = 500` and `DaysSinceLastStay = 100` has Tenure = 400: a loyal repeat guest who first came 500 days ago and returned as recently as 100 days ago.

`DaysSinceFirstStay` is dropped after this step (its information is now captured by `DaysSinceLastStay` (recency) and `Tenure` (loyalty span)). `DaysSinceCreation` is also dropped: it has a correlation of 0.9996 with `DaysSinceFirstStay` and is an administrative timestamp, not a customer behaviour.

In [13]:
df['Tenure'] = df['DaysSinceFirstStay'] - df['DaysSinceLastStay']

df = df.drop(columns=['DaysSinceFirstStay', 'DaysSinceCreation'])

print(f'Tenure: median={df["Tenure"].median():.1f}, max={df["Tenure"].max():.1f}')
print(f'Columns remaining: {df.shape[1]}')

Tenure: median=0.0, max=1100.0
Columns remaining: 17


### C. Age Bins

Age is binned into five groups and ordinally encoded as integers 1–5. Binning was chosen over raw age because: age within a decade is unlikely to produce meaningfully different marketing responses, what matters is the life stage, not the exact year. 

`Age` is replaced by `AgeBin` and is used as a descriptor in cluster profiling, not as a clustering input. The rationale for this is documented in section 3.4.

In [14]:
df['AgeBin'] = pd.cut(
    df['Age'],
    bins=[17, 29, 39, 49, 59, 100],
    labels=[1, 2, 3, 4, 5]
).astype(float)

# drop raw age — AgeBin is its replacement
df = df.drop(columns=['Age'])

print('AgeBin distribution:')
display(pd.DataFrame({
    'Label': ['18–29','30–39','40–49','50–59','60+'],
    'Count': df['AgeBin'].value_counts().sort_index().values,
    'Pct': (df['AgeBin'].value_counts(normalize=True).sort_index()*100).round(1).values
}))

AgeBin distribution:


,Label,Count,Pct
0,18–29,6620,10.9
1,30–39,11457,18.8
2,40–49,16357,26.9
3,50–59,13909,22.8
4,60+,12560,20.6


## 3.4 Format Data

This section prepares the final modelling input and the profile dataset used for cluster interpretation.

### Feature selection for clustering

An exhaustive search was run over all subsets of candidate variables (size 3–6), all combinations of MinMaxScaler and StandardScaler, and K values from 3 to 6. Only results where every cluster contained at least 5% of customers were considered valid.

The best result across all tested combinations:

`DaysSinceLastStay` + `TotalRevenue` + `Tenure` 

Variables excluded from the model and why:

- **`BookingsCheckedIn`**: after merging split profiles, 91% of customers have exactly 1 checked-in booking. Near-zero variance, adds nothing.
- **`AverageLeadTime`**: consistently weakened cluster separation in every tested combination. Retained as a descriptor.
- **`AgeBin`**: Spearman correlation with all behavioural variables is at most 0.18. Retained as a descriptor.
- **SR flags**: binary variables with low prevalence. Retained as descriptors.
- **`RoomNights`**, **`PersonsNights`**: Spearman correlation with `TotalRevenue` is 0.74 and 0.70. Redundant after `TotalRevenue` is included.
- **`OtherRevenueRatio`**, **`RevenuePerRoomNight`**, **`PersonsPerRoomNight`**: all derived features tested. None improved cluster scores when combined with the three winning variables.
- **`Nationality`**, **`MarketSegment`**, **`DistributionChannel`**: categorical descriptors retained in `df_profile` for segment profiling.

### A. Group Rare Nationalities

156 of 188 nationalities represent less than 1% of customers each. These are grouped into `Other` so that cluster profiles are readable. This applies to `df_profile` only, nationality is not a clustering input.

In [15]:
nationality_freq = df['Nationality'].value_counts(normalize=True)
rare_nationalities = nationality_freq[nationality_freq < 0.01].index

df['Nationality'] = df['Nationality'].apply(
    lambda x: 'Other' if x in rare_nationalities else x
)

print(f'Nationality categories after grouping: {df["Nationality"].nunique()}')

Nationality categories after grouping: 18


### B. Save Profile Dataset

`df_profile` is saved here.

In [16]:
df_profile = df.copy()

print(f'df_profile shape: {df_profile.shape}')
print(f'Columns: {df_profile.columns.tolist()}')

df_profile shape: (60903, 17)
Columns: ['AverageLeadTime', 'BookingsCanceled', 'BookingsNoShowed', 'BookingsCheckedIn', 'PersonsNights', 'RoomNights', 'DaysSinceLastStay', 'Nationality', 'MarketSegment', 'DistributionChannel', 'SRHighFloor', 'SRKingSizeBed', 'SRTwinBed', 'SRQuietRoom', 'TotalRevenue', 'Tenure', 'AgeBin']


### C. Select Modelling Variables

The three variables selected for clustering are extracted into `X_unscaled`. All other variables remain in `df_profile` for post-clustering interpretation.

In [17]:
model_features = [
    'DaysSinceLastStay',
    'TotalRevenue',
    'Tenure'
]

X_unscaled = df_profile[model_features].copy()

print(f'Modelling dataset shape: {X_unscaled.shape}')
display(X_unscaled.describe().round(2))

Modelling dataset shape: (60903, 3)


,DaysSinceLastStay,TotalRevenue,Tenure
count,60903.00,60903.00,60903.00
mean,526.78,478.57,5.79
std,303.09,376.97,53.39
min,0.00,0.00,0.00
25%,254.00,242.40,0.00
50%,528.00,384.00,0.00
75%,799.00,585.08,0.00
max,1104.00,2490.36,1100.00


### D. Scale

`MinMaxScaler` maps all three variables to [0, 1]. StandardScaler was tested and consistently produced lower silhouette scores, so minmax was kept.

No PCA is applied. With three variables there is no dimensionality to reduce.

In [18]:
scaler = MinMaxScaler()

X_model = pd.DataFrame(
    scaler.fit_transform(X_unscaled),
    columns=X_unscaled.columns,
    index=X_unscaled.index
)

print(f'Modelling dataset shape: {X_model.shape}')
print(f'Missing values: {X_model.isnull().sum().sum()}')
display(X_model.describe().round(3))

Modelling dataset shape: (60903, 3)
Missing values: 0


,DaysSinceLastStay,TotalRevenue,Tenure
count,60903.000,60903.000,60903.000
mean,0.477,0.192,0.005
std,0.275,0.151,0.049
min,0.000,0.000,0.000
25%,0.230,0.097,0.000
50%,0.478,0.154,0.000
75%,0.724,0.235,0.000
max,1.000,1.000,1.000


## Output

| Object | Description | Shape |
|--------|-------------|-------|
| `X_model` | MinMaxScaled clustering input: `DaysSinceLastStay`, `TotalRevenue`, `Tenure` | (60,903, 3) |
| `X_unscaled` | Same variables unscaled, used alongside `df_profile` for cluster interpretation | (60,903, 3) |
| `df_profile` | Full unscaled dataset with all variables including categorical descriptors | (60,903, 18) |

The final modelling dataset contains **active adult consolidated customers**: ghost customers removed, exact duplicate profiles removed, split profiles consolidated, and customers under 18 excluded as a business decision.

In [19]:
# saving the datasets
path = "../data/processed"
os.makedirs("../data/processed", exist_ok=True)

X_model.to_csv(f"{path}/X_model.csv", index=True)
X_unscaled.to_csv(f"{path}/X_unscaled.csv", index=True)
df_profile.to_csv(f"{path}/df_profile.csv", index=True)

# save scaler and feature list
joblib.dump(scaler, f"{path}/minmax_scaler.pkl")
joblib.dump(model_features, f"{path}/model_features.pkl")

['../data/processed/model_features.pkl']